# GPU Reduction K-Means: dimension sweep

We use one thread per point for assignment and shared-memory reductions over centroid coordinates and cluster counts for centroid updates. We reuse canonical initial rows and select D=2, 4 and 9 without changing N or K.

Run setup and configuration before the batch. See [readme.md](readme.md), [implementation design](docs/IMPLEMENTATION_DESIGN.md) and [results notes](docs/EXPERIMENTS_AND_RESULTS.md).


In [ ]:
import os
import re
import math
import time
import json
import platform
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import numba
from numba import cuda


## Data layout

Drive contains `kmeans_data/rows*/` and full-9D canonical CSVs under `results_gpu_atomic_v3/`; the optional second canonical folder is supported.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Google Colab.")

if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/kmeans_data")
else:
    DATA_ROOT = Path("./kmeans_data")

OUT_DIR = DATA_ROOT / "results_gpu_reduction_v3_aligned_dimension_sweep"
ASSIGNMENTS_DIR = OUT_DIR / "assignments"
CENTROIDS_DIR = OUT_DIR / "centroids"
HISTORY_DIR = OUT_DIR / "history"

OUT_DIR.mkdir(parents=True, exist_ok=True)
ASSIGNMENTS_DIR.mkdir(parents=True, exist_ok=True)
CENTROIDS_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = "all"

DIAGNOSTIC_CASES = [
    (10_000, 3),
    (1_000_000, 3),
    (5_000_000, 9),
    (10_000_000, 17),
]

TARGET_ROWS = 10_000
TARGET_CLUSTERS = 3
TARGET_CSV_COLS = None

RUN_SIZES = None

TOL = 1e-4

ASSIGNMENT_CHANGE_FRACTION_TOL = 1e-4

ASSIGNMENT_STABLE_ITERS = 3

MAX_ITER = 400
SEED = 42

SKIP_COMPLETED = True
CONTINUE_ON_ERROR = True

SAVE_ASSIGNMENTS = True
SAVE_INITIAL_CENTROIDS = True
SAVE_FINAL_CENTROIDS = True
SAVE_ITERATION_HISTORY = True

REFERENCE_INITIAL_CENTROIDS_DIR = DATA_ROOT / "results_gpu_atomic_v3"

REQUIRE_REFERENCE_INITIAL_CENTROIDS = True

print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR  :", OUT_DIR)
print("RUN_MODE :", RUN_MODE)
print(
    "Assignment change tolerance:",
    ASSIGNMENT_CHANGE_FRACTION_TOL,
    f"({ASSIGNMENT_CHANGE_FRACTION_TOL * 100:.4f}%)"
)

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Could not find dataset folder: {DATA_ROOT}"
    )

from pathlib import Path

# Select the first D named features; each dataset keeps all its rows.
FEATURE_DIMS = [2, 4, 9]

SWEEP_CASES = None

CANONICAL_INIT_EXTRA_FOLDERS = ["results_gpu_atomic_v3 2"]

def resolve_canonical_initial_path(primary_directory, base_tag):
    primary = Path(primary_directory)
    folders = [primary] + [primary.parent / name for name in CANONICAL_INIT_EXTRA_FOLDERS]
    filename = f"initial_centroids_{base_tag}.csv"
    candidates = [folder / filename for folder in folders]
    for path in candidates:
        if path.exists():
            return path
    searched = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Full-9D canonical initialization missing; searched: {searched}")

def select_feature_columns(available, dimension):
    if isinstance(dimension, (bool, np.bool_)) or not isinstance(dimension, (int, np.integer)):
        raise ValueError("Feature dimensions must be integers")
    if not 1 <= dimension <= 9:
        raise ValueError(f"D={dimension}: existing data supports dimensions 1 through 9")
    selected = [f"col_{i}" for i in range(1, int(dimension) + 1)]
    missing = [c for c in selected if c not in available]
    if missing:
        raise ValueError(f"Missing feature columns: {missing}")
    return selected

def dimension_tag(base_tag, dimension):
    return f"{base_tag}_D{int(dimension)}"

if not FEATURE_DIMS or len(set(FEATURE_DIMS)) != len(FEATURE_DIMS):
    raise ValueError("FEATURE_DIMS must be nonempty and contain no duplicates")
for _dimension in FEATURE_DIMS:
    select_feature_columns([f"col_{i}" for i in range(1, 10)], _dimension)
print("Dimension sweep:", FEATURE_DIMS, "| (N, K) filter:", SWEEP_CASES)


In [ ]:
DATASET_PATTERN = re.compile(
    r"^synthetic_data_(\d+)_rows_(\d+)_cols_(\d+)_clusters\.csv$"
)

ROW_FOLDER_PATTERN = re.compile(r"^rows(\d+)$")

def discover_datasets(data_root):
    'Scan folders named exactly rows<number>, e.g.'
    records = []

    for folder in sorted(data_root.iterdir()):
        if not folder.is_dir():
            continue

        folder_match = ROW_FOLDER_PATTERN.match(folder.name)
        if folder_match is None:
            continue

        folder_rows = int(folder_match.group(1))

        for csv_path in sorted(folder.glob("*.csv")):
            match = DATASET_PATTERN.match(csv_path.name)

            if match is None:
                continue

            filename_rows = int(match.group(1))
            csv_cols = int(match.group(2))
            clusters = int(match.group(3))

            records.append({
                "rows": filename_rows,
                "csv_cols": csv_cols,
                "clusters": clusters,
                "folder": folder.name,
                "folder_rows": folder_rows,
                "filename": csv_path.name,
                "path": str(csv_path),
                "folder_matches_filename_rows": (
                    folder_rows == filename_rows
                ),
            })

    if not records:
        raise FileNotFoundError(
            f"No matching datasets found under rows<number> folders in {data_root}"
        )

    catalog = (
        pd.DataFrame(records)
        .sort_values(["rows", "clusters", "csv_cols"])
        .reset_index(drop=True)
    )

    mismatches = catalog[
        ~catalog["folder_matches_filename_rows"]
    ]

    if len(mismatches):
        print("WARNING: folder/filename row-count mismatches:")
        display(
            mismatches[
                ["folder", "filename", "folder_rows", "rows"]
            ]
        )

    return catalog

dataset_catalog = discover_datasets(DATA_ROOT)

print(
    f"Discovered {len(dataset_catalog)} dataset files "
    f"across {dataset_catalog['folder'].nunique()} row folders."
)

display(
    dataset_catalog[
        ["folder", "rows", "csv_cols", "clusters", "filename"]
    ]
)


## Dataset selection

`RUN_MODE = "all"` and `SWEEP_CASES = None` cover all discovered files at each configured D.


In [ ]:
def select_dataset(
    catalog,
    target_rows,
    target_clusters,
    target_csv_cols=None
):
    matches = catalog[
        (catalog["rows"] == target_rows)
        & (catalog["clusters"] == target_clusters)
    ]

    if target_csv_cols is not None:
        matches = matches[
            matches["csv_cols"] == target_csv_cols
        ]

    if len(matches) == 0:
        raise ValueError(
            "No dataset matched the requested configuration.\n\n"
            + catalog[
                ["folder", "rows", "csv_cols", "clusters", "filename"]
            ].to_string(index=False)
        )

    if len(matches) > 1:
        raise ValueError(
            "More than one dataset matched. "
            "Set TARGET_CSV_COLS to disambiguate.\n\n"
            + matches.to_string(index=False)
        )

    return matches.iloc[0]

selected = None
DATASET_PATH = None

if RUN_MODE == "single":
    selected = select_dataset(
        dataset_catalog,
        TARGET_ROWS,
        TARGET_CLUSTERS,
        TARGET_CSV_COLS
    )

    DATASET_PATH = Path(selected["path"])

    print("Selected:", DATASET_PATH)
else:
    print(
        "Batch mode enabled: every matching CSV inside every "
        "rows<number> folder will be processed."
    )


In [ ]:
LABEL_NAMES = {
    "true_label", "label", "labels",
    "cluster", "clusters", "cluster_id",
    "target", "class", "y"
}

ID_NAMES = {"id", "index", "idx", "row", "row_id"}

def load_dataset_for_kmeans(path, selected_meta, feature_dim=9):
    df = pd.read_csv(path)

    id_candidates = [
        c for c in df.columns
        if str(c).strip().lower() in ID_NAMES
    ]
    id_col = id_candidates[0] if id_candidates else None

    if id_col is not None:
        ids = df[id_col].astype(str).to_numpy()
    else:
        ids = np.arange(len(df)).astype(str)

    label_candidates = [
        c for c in df.columns
        if str(c).strip().lower() in LABEL_NAMES
    ]
    label_col = label_candidates[0] if label_candidates else None

    true_label = (
        df[label_col].to_numpy()
        if label_col is not None
        else None
    )

    excluded = {id_col, label_col}

    feature_cols = [
        c for c in df.columns
        if c not in excluded
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    if not feature_cols:
        raise ValueError(
            f"No numeric feature columns found in {path.name}"
        )

    feature_cols = select_feature_columns(feature_cols, feature_dim)
    df_data = df[feature_cols]

    N = len(df_data)
    D = len(feature_cols)
    K = int(selected_meta["clusters"])

    print("\nDataset verification")
    print("--------------------")
    print("File                     :", path.name)
    print("Actual rows              :", N)
    print("Actual total CSV columns :", len(df.columns))
    print("ID column                :", id_col)
    print("Label column excluded    :", label_col)
    print("K-Means feature count D  :", D)
    print("Cluster count K          :", K)
    print("Feature columns          :", feature_cols)

    if N != int(selected_meta["rows"]):
        print("WARNING: actual row count differs from filename metadata.")

    if len(df.columns) != int(selected_meta["csv_cols"]):
        print(
            "NOTE: actual CSV column count differs from filename metadata. "
            "A generated label column may explain the difference."
        )

    return (
        df,
        ids,
        df_data,
        true_label,
        feature_cols,
        N,
        D,
        K,
        id_col,
        label_col,
    )

if RUN_MODE == "single":
    (
        df,
        ids,
        df_data,
        true_label,
        d_dims,
        N1,
        D1,
        K1,
        ID_COLUMN_FOUND,
        LABEL_COLUMN_FOUND,
    ) = load_dataset_for_kmeans(
        DATASET_PATH,
        selected
    )

    display(df_data.head())


In [ ]:
@cuda.jit
def calculate_clusters(
    data,
    centroids,
    assign,
    changed_count
):
    'One CUDA thread processes one point and assigns it to the nearest centroid.'

    i = cuda.grid(1)

    row = data.shape[0]
    dim = data.shape[1]
    clusters = centroids.shape[0]

    th_id = cuda.threadIdx.x
    block_size = cuda.blockDim.x

    shared_centroids = cuda.shared.array(
        shape=(max_k, max_d),
        dtype=np.float64
    )

    shared_changed = cuda.shared.array(
        shape=256,
        dtype=np.int32
    )

    idx = th_id
    total = clusters * dim

    while idx < total:
        k = idx // dim
        j = idx % dim
        shared_centroids[k, j] = centroids[k, j]
        idx += block_size

    cuda.syncthreads()

    local_changed = 0

    if i < row:
        best_cluster = 0
        best_distance = math.inf

        for k in range(clusters):
            distance = 0.0

            for j in range(dim):
                diff = data[i, j] - shared_centroids[k, j]
                distance += diff * diff

            if distance < best_distance:
                best_distance = distance
                best_cluster = k

        old_cluster = assign[i]

        if old_cluster != best_cluster:
            local_changed = 1

        assign[i] = best_cluster

    shared_changed[th_id] = local_changed
    cuda.syncthreads()

    stride = block_size // 2

    while stride > 0:
        if th_id < stride:
            shared_changed[th_id] += shared_changed[th_id + stride]

        cuda.syncthreads()
        stride //= 2

    if th_id == 0 and shared_changed[0] > 0:
        cuda.atomic.add(
            changed_count,
            0,
            shared_changed[0]
        )


In [ ]:
@cuda.jit
def cal_centroid_sums(
    data,
    assignments,
    centroid_sums
):
    'Reduction approach from the team notebook.'
    th_id = cuda.threadIdx.x
    block_id = cuda.blockIdx.x
    block_size = cuda.blockDim.x

    N = data.shape[0]
    D = data.shape[1]

    k = block_id // D
    j = block_id % D

    if k >= centroid_sums.shape[0]:
        return

    shared_sum = cuda.shared.array(
        shape=256,
        dtype=np.float64
    )

    local_sum = 0.0
    i = th_id

    while i < N:
        if assignments[i] == k:
            local_sum += data[i, j]
        i += block_size

    shared_sum[th_id] = local_sum
    cuda.syncthreads()

    stride = block_size // 2

    while stride > 0:
        if th_id < stride:
            shared_sum[th_id] += shared_sum[th_id + stride]

        cuda.syncthreads()
        stride //= 2

    if th_id == 0:
        centroid_sums[k, j] = shared_sum[0]

@cuda.jit
def cal_cluster_count(
    assignments,
    cluster_counts
):
    'One reduction block per cluster.'
    th_id = cuda.threadIdx.x
    block_id = cuda.blockIdx.x
    block_size = cuda.blockDim.x

    N = assignments.shape[0]
    K = cluster_counts.shape[0]

    if block_id >= K:
        return

    shared_count = cuda.shared.array(
        shape=256,
        dtype=np.int32
    )

    local_count = 0
    i = th_id

    while i < N:
        if assignments[i] == block_id:
            local_count += 1
        i += block_size

    shared_count[th_id] = local_count
    cuda.syncthreads()

    stride = block_size // 2

    while stride > 0:
        if th_id < stride:
            shared_count[th_id] += shared_count[th_id + stride]

        cuda.syncthreads()
        stride //= 2

    if th_id == 0:
        cluster_counts[block_id] = shared_count[0]


## Centroid accumulation

We compare global atomic updates with shared-memory scan reductions using the same assignment and convergence rules.


In [ ]:
@cuda.jit
def update_centroids(centroid_sums, cluster_counts, centroids):
    'updated centroid with mean'

    block_id = cuda.blockIdx.x
    th_id = cuda.threadIdx.x

    K = centroids.shape[0]
    D = centroids.shape[1]

    if block_id >= K or th_id >= D:
        return

    count = cluster_counts[block_id]

    if count > 0:

        centroids[block_id, th_id] = (centroid_sums[block_id, th_id] / count)


In [ ]:
import time
import pandas as pd


In [ ]:
def run_kmeans_reduction(
    df_data,
    K,
    max_iter=MAX_ITER,
    tol=TOL,
    assignment_fraction_tol=ASSIGNMENT_CHANGE_FRACTION_TOL,
    assignment_stable_iters=ASSIGNMENT_STABLE_ITERS,
    initial_indices=None,
    initial_centroids=None,
    seed=SEED,
    verbose=True
):
    'Reduction-based GPU K-Means using the same convergence/timing protocol'
    h_data = np.ascontiguousarray(
        df_data.to_numpy(dtype=np.float64)
    )

    N, D = h_data.shape

    if K > max_k:
        raise ValueError(f"K={K} exceeds max_k={max_k}")

    if D > max_d:
        raise ValueError(f"D={D} exceeds max_d={max_d}")

    if verbose:
        print(
            f"--- Running K-Means REDUCTION V3 "
            f"(N={N}, D={D}, K={K}) ---"
        )

    if initial_centroids is not None:
        h_centroids = np.ascontiguousarray(
            np.asarray(initial_centroids, dtype=np.float64)
        )

        if h_centroids.shape != (K, D):
            raise ValueError(
                f"initial_centroids shape {h_centroids.shape} "
                f"does not match expected {(K, D)}"
            )

        initial_indices_used = None
    else:
        if initial_indices is None:
            rng = np.random.default_rng(seed)
            initial_indices = rng.choice(
                N,
                K,
                replace=False
            )

        initial_indices = np.asarray(
            initial_indices,
            dtype=np.int64
        )

        h_centroids = h_data[initial_indices].copy()
        initial_indices_used = initial_indices.copy()

    initial_centroids_used = h_centroids.copy()
    h_assign = np.full(N, -1, dtype=np.int32)

    d_data = cuda.to_device(h_data)
    d_centroids = cuda.to_device(h_centroids)
    d_assign = cuda.to_device(h_assign)

    blocks_per_grid_points = math.ceil(
        N / THREADS_PER_BLOCK
    )

    h_zero_changed = np.zeros(1, dtype=np.int32)
    d_changed_count = cuda.to_device(h_zero_changed)

    warm_n = min(N, 4096)

    d_warm_data = cuda.to_device(
        np.ascontiguousarray(h_data[:warm_n])
    )
    d_warm_centroids = cuda.to_device(
        initial_centroids_used.copy()
    )
    d_warm_assign = cuda.to_device(
        np.full(warm_n, -1, dtype=np.int32)
    )
    d_warm_sums = cuda.to_device(
        np.zeros((K, D), dtype=np.float64)
    )
    d_warm_counts = cuda.to_device(
        np.zeros(K, dtype=np.int32)
    )
    d_warm_changed = cuda.to_device(
        np.zeros(1, dtype=np.int32)
    )

    warm_blocks = math.ceil(
        warm_n / THREADS_PER_BLOCK
    )

    warmup_start = time.perf_counter()

    calculate_clusters[
        warm_blocks,
        THREADS_PER_BLOCK
    ](
        d_warm_data,
        d_warm_centroids,
        d_warm_assign,
        d_warm_changed
    )

    cal_centroid_sums[
        K * D,
        THREADS_PER_BLOCK
    ](
        d_warm_data,
        d_warm_assign,
        d_warm_sums
    )

    cal_cluster_count[
        K,
        THREADS_PER_BLOCK
    ](
        d_warm_assign,
        d_warm_counts
    )

    update_centroids[
        K,
        D
    ](
        d_warm_sums,
        d_warm_counts,
        d_warm_centroids
    )

    cuda.synchronize()

    warmup_time = (
        time.perf_counter() - warmup_start
    )

    if verbose:
        print(
            f"CUDA/Numba warm-up: {warmup_time:.4f} s "
            "(excluded)"
        )

    start_time = time.perf_counter()

    cent_shift = math.inf
    changed = N
    changed_fraction = 1.0

    converged = False
    convergence_reason = "max_iterations"
    stable_fraction_streak = 0
    history = []

    for iteration in range(max_iter):
        old_centroids = h_centroids.copy()

        d_centroid_sums = cuda.to_device(
            np.zeros((K, D), dtype=np.float64)
        )

        d_cluster_counts = cuda.to_device(
            np.zeros(K, dtype=np.int32)
        )

        d_changed_count.copy_to_device(
            h_zero_changed
        )

        calculate_clusters[
            blocks_per_grid_points,
            THREADS_PER_BLOCK
        ](
            d_data,
            d_centroids,
            d_assign,
            d_changed_count
        )

        cuda.synchronize()

        changed = int(
            d_changed_count.copy_to_host()[0]
        )

        changed_fraction = (
            changed / float(N)
        )

        cal_centroid_sums[
            K * D,
            THREADS_PER_BLOCK
        ](
            d_data,
            d_assign,
            d_centroid_sums
        )

        cuda.synchronize()

        cal_cluster_count[
            K,
            THREADS_PER_BLOCK
        ](
            d_assign,
            d_cluster_counts
        )

        cuda.synchronize()

        update_centroids[
            K,
            D
        ](
            d_centroid_sums,
            d_cluster_counts,
            d_centroids
        )

        cuda.synchronize()

        d_centroids.copy_to_host(
            h_centroids
        )

        cent_shift = float(
            np.max(
                np.sqrt(
                    np.sum(
                        (h_centroids - old_centroids) ** 2,
                        axis=1
                    )
                )
            )
        )

        centroid_converged = (
            cent_shift < tol
        )

        assignment_fraction_met = (
            changed_fraction
            <= assignment_fraction_tol
        )

        if assignment_fraction_met:
            stable_fraction_streak += 1
        else:
            stable_fraction_streak = 0

        assignment_stability_met = (
            stable_fraction_streak
            >= assignment_stable_iters
        )

        history.append({
            "iteration": iteration + 1,
            "assignment_changes": changed,
            "assignment_change_fraction": changed_fraction,
            "assignment_change_percent": changed_fraction * 100.0,
            "max_centroid_movement": cent_shift,
            "stable_fraction_streak": stable_fraction_streak,
            "centroid_tolerance_met": centroid_converged,
            "assignment_fraction_tolerance_met": assignment_fraction_met,
            "assignment_stability_met": assignment_stability_met,
        })

        if verbose and (
            iteration < 5
            or assignment_fraction_met
            or (iteration + 1) % 25 == 0
        ):
            print(
                f"iter={iteration + 1:4d} | "
                f"changed={changed:10d} | "
                f"changed%={changed_fraction * 100:.5f}% | "
                f"shift={cent_shift:.8f} | "
                f"stable-streak={stable_fraction_streak}"
            )

        if centroid_converged or assignment_stability_met:
            converged = True

            if centroid_converged and assignment_stability_met:
                convergence_reason = "both"
            elif assignment_stability_met:
                convergence_reason = "assignment_fraction_stable"
            else:
                convergence_reason = "centroid_tolerance"

            break

    cuda.synchronize()

    exec_time = (
        time.perf_counter() - start_time
    )

    d_assign.copy_to_host(
        h_assign
    )

    if verbose:
        print(f"Timed execution      : {exec_time:.4f} s")
        print(f"Warm-up excluded     : {warmup_time:.4f} s")
        print(f"Iterations           : {iteration + 1}")
        print(f"Converged            : {converged}")
        print(f"Reason               : {convergence_reason}")
        print(f"Final changes        : {changed}")
        print(
            f"Final change fraction : "
            f"{changed_fraction:.8e}"
        )
        print(f"Final shift          : {cent_shift:.8f}")

    return {
        "assignments": h_assign,
        "centroids": h_centroids,
        "initial_centroids": initial_centroids_used,
        "initial_indices": initial_indices_used,
        "iterations": iteration + 1,
        "execution_time": float(exec_time),
        "warmup_time": float(warmup_time),
        "max_movement": cent_shift,
        "assignment_changes": int(changed),
        "assignment_change_fraction": float(changed_fraction),
        "stable_fraction_streak": int(stable_fraction_streak),
        "converged": converged,
        "convergence_reason": convergence_reason,
        "history": pd.DataFrame(history),
    }


## Saved results

Each `_D<dimension>` case has final results, initial/final centroid tables, assignments and an iteration history.


In [ ]:
max_k = 64
max_d = 32
THREADS_PER_BLOCK = 256

try:
    device = cuda.get_current_device()
    gpu_name = device.name

    if isinstance(gpu_name, bytes):
        gpu_name = gpu_name.decode()

    compute_capability = ".".join(
        map(str, device.compute_capability)
    )
except Exception:
    gpu_name = "unknown"
    compute_capability = "unknown"

ENV = {
    "feature_dimensions": FEATURE_DIMS,
    "sweep_cases": SWEEP_CASES,
    "feature_selection": "first D named col_* features",
    "implementation": "gpu_reduction_numba_cuda_v3_aligned",
    "gpu_name": gpu_name,
    "compute_capability": compute_capability,
    "platform": platform.platform(),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "numba": numba.__version__,
    "threads_per_block": THREADS_PER_BLOCK,
    "max_k": max_k,
    "max_d": max_d,
    "centroid_tolerance": TOL,
    "assignment_change_fraction_tolerance":
        ASSIGNMENT_CHANGE_FRACTION_TOL,
    "assignment_stable_iterations":
        ASSIGNMENT_STABLE_ITERS,
    "max_iterations": MAX_ITER,
    "seed": SEED,
    "jit_warmup_excluded_from_timing": True,
    "canonical_initial_centroids_dir": str(REFERENCE_INITIAL_CENTROIDS_DIR),
    "data_root": str(DATA_ROOT),
}

with open(
    OUT_DIR / "environment.json",
    "w"
) as fh:
    json.dump(
        ENV,
        fh,
        indent=2
    )

ENV


## Output helpers

We calculate assigned distances in chunks after timing, using the labels retained at the final iteration.


In [ ]:
def dataset_tag_from_path(path):
    return Path(path).stem

def calculate_assigned_distances(
    X,
    centroids,
    assignments,
    chunk_size=500_000
):
    'Distance from each point to its assigned final centroid.'
    X = np.asarray(X, dtype=np.float64)
    assignments = np.asarray(assignments, dtype=np.int32)

    distances = np.empty(len(X), dtype=np.float64)

    for start in range(0, len(X), chunk_size):
        end = min(start + chunk_size, len(X))

        a = assignments[start:end]
        diff = X[start:end] - centroids[a]

        distances[start:end] = np.sqrt(
            np.sum(diff * diff, axis=1)
        )

    return distances

def calculate_wcss_from_distances(distances):
    return float(
        np.sum(
            np.asarray(distances, dtype=np.float64) ** 2
        )
    )

def initial_centroid_sha256(C):
    arr = np.ascontiguousarray(
        np.asarray(C, dtype=np.float64)
    )
    return hashlib.sha256(arr.tobytes()).hexdigest()

def _read_initial_centroid_csv(path, K, D, feature_cols=None):
    'Read the canonical Atomic-V3 centroid file.'
    df = pd.read_csv(path)

    if "cluster" in df.columns:
        df = df.drop(columns=["cluster"])

    if feature_cols is not None:
        missing = [c for c in feature_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Canonical centroid columns missing: {missing}")
        df = df[list(feature_cols)]
    else:
        df = df.select_dtypes(include=[np.number])

    arr = df.to_numpy(dtype=np.float64)

    if arr.shape != (K, D):
        raise ValueError(
            f"{path} has centroid shape {arr.shape}; expected {(K, D)}"
        )

    return np.ascontiguousarray(arr, dtype=np.float64)

def find_reference_initial_centroids(tag, K, D, feature_cols=None):
    'Load selected columns from an existing full-9D canonical file.'
    if REFERENCE_INITIAL_CENTROIDS_DIR is None:
        raise ValueError("Dimension sweeps require a canonical initial-centroid directory")
    path = resolve_canonical_initial_path(REFERENCE_INITIAL_CENTROIDS_DIR, tag)

    arr = _read_initial_centroid_csv(
        path,
        K,
        D,
        feature_cols=feature_cols
    )

    print("Using canonical initial centroids:", path)
    print("Initial centroid SHA256:", initial_centroid_sha256(arr))

    return arr

def save_run_outputs(
    path,
    meta,
    ids,
    feature_cols,
    true_label,
    X,
    result
):
    tag = dimension_tag(dataset_tag_from_path(path), len(feature_cols))

    assignments = result["assignments"]
    centroids = result["centroids"]

    distances = calculate_assigned_distances(
        X,
        centroids,
        assignments
    )

    wcss = calculate_wcss_from_distances(distances)

    cluster_sizes = np.bincount(
        assignments,
        minlength=len(centroids)
    )

    if SAVE_INITIAL_CENTROIDS:
        init_df = pd.DataFrame(
            result["initial_centroids"],
            columns=feature_cols
        )
        init_df.insert(
            0,
            "cluster",
            np.arange(len(init_df))
        )
        init_df.to_csv(
            OUT_DIR / f"initial_centroids_{tag}.csv",
            index=False
        )

    if SAVE_FINAL_CENTROIDS:
        cent_df = pd.DataFrame(
            centroids,
            columns=feature_cols
        )
        cent_df.insert(
            0,
            "cluster",
            np.arange(len(cent_df))
        )
        cent_df["cluster_size"] = cluster_sizes

        cent_df.to_csv(
            CENTROIDS_DIR / f"final_centroids_{tag}.csv",
            index=False
        )

    assignment_path = ASSIGNMENTS_DIR / f"assignments_{tag}.csv"

    if SAVE_ASSIGNMENTS:
        out = pd.DataFrame({
            "ID": ids,
            "cluster": assignments,
            "distance": distances,
        })

        if true_label is not None:
            out["true_label"] = true_label

        out.to_csv(
            assignment_path,
            index=False
        )

    if SAVE_ITERATION_HISTORY:
        result["history"].to_csv(
            HISTORY_DIR / f"history_{tag}.csv",
            index=False
        )

    final_row = {
        "Implementation": "gpu_reduction_v3_aligned",
        "Initial centroid source": str(resolve_canonical_initial_path(REFERENCE_INITIAL_CENTROIDS_DIR, dataset_tag_from_path(path))),
        "Initial centroid SHA256": initial_centroid_sha256(result["initial_centroids"]),
        "Dataset": tag,
        "Source dataset": dataset_tag_from_path(path),
        "Feature columns": json.dumps(list(feature_cols)),
        "Rows": len(X),
        "Features": X.shape[1],
        "K": len(centroids),
        "Execution time": result["execution_time"],
        "Warmup time": result["warmup_time"],
        "Warmup excluded": True,
        "Iterations": result["iterations"],
        "Per iteration ms": (
            result["execution_time"]
            / result["iterations"]
            * 1000.0
        ),
        "Converged": result["converged"],
        "Convergence reason": result["convergence_reason"],
        "Final assignment changes":
            result["assignment_changes"],
        "Final assignment change fraction":
            result["assignment_change_fraction"],
        "Final assignment change percent":
            result["assignment_change_fraction"] * 100.0,
        "Stable fraction streak":
            result["stable_fraction_streak"],
        "Max movement": result["max_movement"],
        "WCSS": wcss,
        "Cluster sizes": json.dumps(
            cluster_sizes.tolist()
        ),
        "Centroids": json.dumps(
            centroids.astype(float).tolist()
        ),
    }

    pd.DataFrame([final_row]).to_csv(
        OUT_DIR / f"final_results_{tag}.csv",
        index=False
    )

    return final_row, assignment_path


## Batch and resume

We persist the summary after each completed N/K/D case and resume from dimension-specific final-result files.


In [ ]:
SUMMARY_PATH = OUT_DIR / "summary.csv"

def update_summary_csv(new_rows):
    valid_rows = [
        row for row in new_rows
        if row is not None
    ]

    new_df = pd.DataFrame(
        valid_rows
    )

    if SUMMARY_PATH.exists():
        old_df = pd.read_csv(
            SUMMARY_PATH
        )

        if len(new_df):
            combined = pd.concat(
                [old_df, new_df],
                ignore_index=True
            )
        else:
            combined = old_df
    else:
        combined = new_df

    if len(combined):
        combined = combined.drop_duplicates(
            subset=[
                "Dataset",
                "Features",
                "Implementation"
            ],
            keep="last"
        )

        combined = combined.sort_values(
            [
                "Rows",
                "K",
                "Features",
                "Implementation"
            ]
        ).reset_index(drop=True)

    combined.to_csv(
        SUMMARY_PATH,
        index=False
    )

    return combined

def final_result_path_for(path, feature_dim):
    tag = dimension_tag(dataset_tag_from_path(path), feature_dim)

    return (
        OUT_DIR
        / f"final_results_{tag}.csv"
    )

def run_one_dataset(path, meta, feature_dim):
    final_path = final_result_path_for(path, feature_dim)

    if SKIP_COMPLETED and final_path.exists():
        print(
            "SKIP (already completed):",
            path.name
        )

        return (
            pd.read_csv(final_path)
            .iloc[0]
            .to_dict()
        )

    (
        df_full,
        ids_local,
        df_features,
        true_label_local,
        feature_cols,
        N,
        D,
        K,
        id_col,
        label_col,
    ) = load_dataset_for_kmeans(
        path,
        meta,
        feature_dim=feature_dim
    )

    if K > max_k:
        raise ValueError(
            f"K={K} exceeds max_k={max_k}"
        )

    if D > max_d:
        raise ValueError(
            f"D={D} exceeds max_d={max_d}"
        )

    tag = dataset_tag_from_path(
        path
    )

    reference_C0 = (
        find_reference_initial_centroids(
            tag,
            K,
            D,
            feature_cols=feature_cols
        )
    )

    result = run_kmeans_reduction(
        df_data=df_features,
        K=K,
        max_iter=MAX_ITER,
        tol=TOL,
        seed=SEED,
        initial_centroids=reference_C0,
        verbose=True
    )

    final_row, assignment_path = (
        save_run_outputs(
            path=path,
            meta=meta,
            ids=ids_local,
            feature_cols=feature_cols,
            true_label=true_label_local,
            X=df_features.to_numpy(
                dtype=np.float64
            ),
            result=result
        )
    )

    print(
        "Saved final result:",
        final_path
    )

    if SAVE_ASSIGNMENTS:
        print(
            "Saved assignments:",
            assignment_path
        )

    if SAVE_ITERATION_HISTORY:
        print(
            "Saved history:",
            HISTORY_DIR
            / f"history_{dimension_tag(tag, feature_dim)}.csv"
        )

    return final_row


In [ ]:
def select_diagnostic_catalog(
    catalog,
    cases
):
    selected_rows = []

    for rows, k in cases:
        matches = catalog[
            (catalog["rows"] == rows)
            & (catalog["clusters"] == k)
        ]

        if len(matches) != 1:
            raise ValueError(
                f"Expected exactly one dataset for "
                f"rows={rows}, K={k}; found {len(matches)}"
            )

        selected_rows.append(
            matches.iloc[0]
        )

    return pd.DataFrame(
        selected_rows
    ).reset_index(drop=True)

if RUN_MODE == "single":
    catalog_to_run = pd.DataFrame([
        selected
    ])

elif RUN_MODE == "diagnostic":
    catalog_to_run = (
        select_diagnostic_catalog(
            dataset_catalog,
            DIAGNOSTIC_CASES
        )
    )

elif RUN_MODE == "all":
    catalog_to_run = (
        dataset_catalog.copy()
    )

    if RUN_SIZES is not None:
        catalog_to_run = (
            catalog_to_run[
                catalog_to_run["rows"]
                .isin(RUN_SIZES)
            ].copy()
        )

else:
    raise ValueError(
        "RUN_MODE must be one of: "
        "'diagnostic', 'single', 'all'"
    )

if SWEEP_CASES is not None:
    catalog_to_run = catalog_to_run[
        [(int(m["rows"]), int(m["clusters"])) in SWEEP_CASES
         for _, m in catalog_to_run.iterrows()]
    ].copy()

cases = [(m, d) for _, m in catalog_to_run.iterrows() for d in FEATURE_DIMS]
print(f"Run plan: {len(catalog_to_run)} CSV files, {len(cases)} (CSV, D) cases")
display(catalog_to_run)
failures = []
for run_number, (meta_row, feature_dim) in enumerate(cases, start=1):
    path = Path(meta_row["path"])
    print(f"\n[{run_number}/{len(cases)}] {path.name}, D={feature_dim}")
    try:
        row = run_one_dataset(path, meta_row, feature_dim)
        summary = update_summary_csv([row])
        print(f"Progress persisted: {len(summary)} cases in summary.csv")
    except Exception as exc:
        failures.append({"folder": meta_row["folder"], "dataset": path.name,
                         "Features": feature_dim, "error_type": type(exc).__name__, "error": str(exc)})
        print(f"ERROR: {type(exc).__name__}: {exc}")
        pd.DataFrame(failures).to_csv(OUT_DIR / "batch_failures.csv", index=False)
        if not CONTINUE_ON_ERROR:
            raise

if SUMMARY_PATH.exists():
    summary = pd.read_csv(
        SUMMARY_PATH
    )
else:
    summary = pd.DataFrame()

print("\nRUN COMPLETE")
print("Summary:", SUMMARY_PATH)
print("Completed rows:", len(summary))
print("Failures:", len(failures))

display(summary)

if failures:
    display(
        pd.DataFrame(failures)
    )


## Runtime plot

We show convergence time by N, K and D, with JIT warm-up excluded.


In [ ]:
import matplotlib.pyplot as plt

if len(summary):
    plot_df = summary[
        summary["Implementation"]
        == "gpu_reduction_v3_aligned"
    ].copy()

    if len(plot_df):
        fig, ax = plt.subplots(
            figsize=(8, 5)
        )

        for (d, k), group in plot_df.groupby(["Features", "K"]):
            group = group.sort_values(
                "Rows"
            )

            ax.plot(
                group["Rows"],
                group["Execution time"],
                marker="o",
                label=f"D={d}, K={k}"
            )

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Rows")
        ax.set_ylabel(
            "Timed convergence-run time (s)"
        )
        ax.set_title(
            "GPU Reduction K-Means V3 Runtime "
            "(JIT Warm-up Excluded)"
        )
        ax.grid(
            True,
            which="both",
            alpha=0.3
        )
        ax.legend(fontsize=8, ncol=2)

        fig.tight_layout()

        plot_path = (
            OUT_DIR
            / "reduction_v3_total_runtime.png"
        )

        fig.savefig(
            plot_path,
            dpi=150
        )

        print(
            "Saved plot:",
            plot_path
        )

        plt.show()


## Case identity

The original filename identifies N/K; `_D2`, `_D4` or `_D9` identifies the selected feature count. Compare hashes and results only within one N/K/D case.


## Run notes

The full 18-file collection produces 54 cases. Keep the summary, history, assignments and environment file together when exporting.


## Stopping rule

We stop on movement below `1e-4`, or assignment changes of at most `1e-4 × N` for three consecutive iterations, with at most 400 iterations. This permits small residual label changes.
